# Water Meter Data Quality Analysis and Cleaning Pipeline

This notebook implements a comprehensive data cleaning and quality assessment pipeline for water meter reading data. The process includes data transformation, validation, and quality reporting.

## Objective
Clean and standardize water meter data from multiple sources with different languages and formats, ensuring data quality and consistency for downstream analysis.

## Dataset Overview
- **Source**: Water meter readings from multiple regions
- **Languages**: English, Spanish, Italian
- **Key Variables**: Reading values, dates, contract types, validity status

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

## 1. Environment Setup

Import necessary libraries for data manipulation, visualization, and date handling.

In [30]:
df = pd.read_csv('datasets/data.csv')

df.head(3)

,No,Water Meter ID,Reading ID,Reading Value,Reading Date,Previous Reading Value,Previous Reading Date,Reading Frequency,Reader ID,Type of Contract,Reading Validity,Certification on the ERP,Final Billing,Reason for Reading
0,1,IT-WM-001,READ-001,145.50,15/01/2024,120.30,15/12/2023,Monthly,READER-01,Residential,Valid,Yes,125.40,Routine
1,2,IT-WM-002,READ-002,89.75,16/01/2024,75.20,16/12/2023,Monthly,READER-02,Commercial,Valid,Yes,89.75,Routine
2,3,IT-WM-003,READ-003,NaN,17/01/2024,210.45,17/12/2023,Monthly,READER-03,Industrial,Invalid,No,0.00,Missing


## 2. Data Loading and Initial Exploration

Load the raw water meter dataset and perform initial inspection to understand the data structure and identify potential issues.

## 3. Data Type Transformations

Convert raw data into appropriate formats for analysis. This includes standardizing date formats and converting multilingual categorical values to consistent boolean types.

### 3.1 Date Standardization
Transform date columns to a consistent YYYY-MM-DD format for proper temporal analysis.

### 3.2 Boolean Conversions
Standardize multilingual boolean values (True/False, Válido/Inválido, etc.) across different languages and formats.

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   No                        501 non-null    int64  
 1   Water Meter ID            501 non-null    object 
 2   Reading ID                501 non-null    object 
 3   Reading Value             494 non-null    float64
 4   Reading Date              501 non-null    object 
 5   Previous Reading Value    494 non-null    float64
 6   Previous Reading Date     496 non-null    object 
 7   Reading Frequency         501 non-null    object 
 8   Reader ID                 499 non-null    object 
 9   Type of Contract          500 non-null    object 
 10  Reading Validity          501 non-null    object 
 11  Certification on the ERP  500 non-null    object 
 12  Final Billing             484 non-null    float64
 13  Reason for Reading        478 non-null    object 
dtypes: float64

In [32]:
# dates transform
times = ['Reading Date', 'Previous Reading Date']

for i in times:
    df[i] = pd.to_datetime(df[i], errors='coerce')
    df[i] = df[i].dt.strftime('%Y-%m-%d')
    print(df[i].head(3))
    print('\n')

0    2024-01-15
1    2024-01-16
2    2024-01-17
Name: Reading Date, dtype: object


0    2023-12-15
1    2023-12-16
2    2023-12-17
Name: Previous Reading Date, dtype: object




/tmp/ipykernel_3773/3314024531.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[i] = pd.to_datetime(df[i], errors='coerce')
/tmp/ipykernel_3773/3314024531.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[i] = pd.to_datetime(df[i], errors='coerce')


In [33]:
# bool transform
# iba a agregar final billing pero el dataset no cumple con las condiciones del diccionario
# asi que lo voy a dejar como esta
bools = ['Certification on the ERP']
for i in bools:
    df[i] = df[i].str.lower()
    df[i] = df[i].replace({'true': True, 'false': False, '0': False, '1': True})
    df[i] = df[i].astype(bool)
    print(df[i].head(3))
    print('\n')

# vality
# Replace values in 'Reading Validity' column using the replace() method
df['Reading Validity'] = df['Reading Validity'].replace(
    {'Valid': True, 'Invalid': False, 'Válido': True, 'Inválido': False, 'Valido': True, 'Sospetto': False}
)
df['Reading Validity'] = df['Reading Validity'].astype(bool)
print(df['Reading Validity'].head(3))

0    True
1    True
2    True
Name: Certification on the ERP, dtype: bool


0     True
1     True
2    False
Name: Reading Validity, dtype: bool


## 4. Data Quality Assessment

Analyze missing values and data quality issues before implementing the cleaning pipeline.

### 4.0 Droping duplicates

We use pandas funtion to drop the duplicates in the dataframe

In [34]:
df = df.drop_duplicates()

### 4.1 Missing Values Analysis

Identify columns with null values to understand data completeness and guide cleaning strategies.

In [35]:
null_count = {}

for col in df.columns:
    count = df[col].isnull().sum()
    if count > 0:
        null_count[col]=count
for col, count in null_count.items():
    print(col,":",count)


Reading Value : 7
Previous Reading Value : 7
Previous Reading Date : 13
Reader ID : 2
Type of Contract : 1
Final Billing : 17
Reason for Reading : 23


In [36]:
df.head(1)

,No,Water Meter ID,Reading ID,Reading Value,Reading Date,Previous Reading Value,Previous Reading Date,Reading Frequency,Reader ID,Type of Contract,Reading Validity,Certification on the ERP,Final Billing,Reason for Reading
0,1,IT-WM-001,READ-001,145.5,2024-01-15,120.3,2023-12-15,Monthly,READER-01,Residential,True,True,125.4,Routine


In [37]:
start_df = df

## 5. Data Cleaning Pipeline Implementation

### 5.1 Validation Rules Definition

Establish comprehensive validation rules for each column including data types, value ranges, and acceptable categorical values. These rules support multilingual data by mapping values from different languages to standardized formats.

### 5.2 Threshold and Validation Configuration

Define specific thresholds, acceptable ranges, and mapping rules for data standardization. The configuration includes:
- **Numeric variables**: Min/max ranges and filling strategies
- **Categorical variables**: Multilingual value mappings and standardization rules

In [38]:
# rules
validation_rules = {
    # numeric
    "Reading Value": {
        "type": "numeric", 
        "min": 0, 
        "max": 1000, 
        "allow_null": False, 
        "fill_strategy": "median"
    },
    "Previous Reading Value": {
        "type": "numeric",
        "min": 0,
        "max": 1000,
        "allow_null": True,
        "fill_strategy":  "median"
    },
    "Final Billing":{
        "type": "numeric",
        "min": 0.0,
        "max": 1000.0,
        "allow_null": False,
        "fill_strategy":  "median"
    },
    # categorical
    "Type of Contract": {
        "type": "categorical",
        "allowed_values": ["residential", "comercial", "industrial"], 
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            "Residential": "residential",
            "Comercial": "comercial",
            "Industrial": "industrial", 
            "residencial": "residential",
            "residenziale": "residential",
            "commerciale": "comercial",
            "industriale": "industrial",
        }
    },
    "Reading Validity":{
        "type": "categorical",
        "allowed_values": [True, False],  # Change to boolean values
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            "valid": True,
            "invalid": False,
            "suspicious": False,
            "valido": True,
            "invalido": False,
            "sospechoso": False,
            "non valido": False,
            "sospetto": False,
        }
    },
    "Certification on the ERP":{
        "type": "categorical",
        "allowed_values": [True, False],  # Change to boolean values
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            "yes": True,
            "no": False,
            "si": True,
            "y": True,
            "1": True,
            "0": False,
        }
    },
    "Reading Frequency": {
        "type": "categorical", 
        "allowed_values": ["monthly", "bimonthly"],
        "allow_null": True,
        "fill_strategy": "mode",
        "mapping": {
            "Monthly": "monthly",
            "Bimonthly": "bimonthly", 
            "mensual": "monthly",
            "bimestral": "bimonthly",
            "mensile": "monthly",
            "bimestrale": "bimonthly",
        }
    },
}

### 5.3 Validation Functions

Create specialized functions to handle different data types with appropriate validation logic:

In [39]:
def validate_numeric(df, column, rules):
    errors = {}
    # out of range values
    out_of_range = ((df[column] < rules["min"]) | (df[column] > rules["max"])).sum()
    if out_of_range > 0:
        df[column] = df[column].clip(lower=rules["min"], upper=rules["max"])
    errors["out_of_range_n"] = out_of_range


    # null operation
    null_before = df[column].isnull().sum()
    if null_before > 0 and not rules["allow_null"]:
        if rules["fill_strategy"] == "median":
            df[column] = df[column].fillna(df[column].median())
        elif rules["fill_strategy"] == "mode":
            df[column] = df[column].fillna(df[column].mode()[0])
    errors["null_values_n"] = null_before
    return df, errors

**Numeric Validation Function**

Handles numeric columns by:
- Clipping values to defined min/max ranges
- Filling null values using median or mode strategies
- Tracking validation errors for reporting

In [40]:
def validate_categoric(df, column, rules):
    errors = {}
    # Preprocess categorical values: convert to lowercase and remove accents
    df[column] = df[column].astype(str).str.lower().str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    
    # using the map
    if "mapping" in rules:
        df[column] = df[column].map(rules["mapping"]).fillna(df[column])

    # null values
    null_before = df[column].isnull().sum()
    if null_before > 0 and not rules["allow_null"]:
        df[column] = df[column].fillna("NaN")
    errors["null_values_c"]=null_before

    # validate against allowed values
    invalid_values = ~df[column].isin(rules["allowed_values"])
    invalid_count = invalid_values.sum()
    
    if invalid_count > 0:
        # Replace invalid values with mode or first allowed value
        mode_val = df[column].mode()
        replacement = mode_val[0] if not mode_val.empty else rules["allowed_values"][0]
        df.loc[invalid_values, column] = replacement
        
    errors["invalid_values_c"] = invalid_count
    return df, errors

**Categorical Validation Function**

Handles categorical columns by:
- Applying multilingual value mappings to standardize terms
- Replacing invalid values with mode or default values
- Managing null values according to column-specific rules

### 5.4 Main Transformation Pipeline

Orchestrate the entire cleaning process by applying validation rules to all columns systematically.

In [41]:
def transform_df(df,rules):
    error_report = {}
    for column, rule in rules.items():
        if column not in df.columns:
            continue
        if rule["type"] == "numeric":
            df, error = validate_numeric(df, column, rule)
            error_report[column] = error
        elif rule["type"] == "categorical":
            df, error = validate_categoric(df, column, rule)
            error_report[column] = error
    return df, error_report


### 5.5 Pipeline Execution and Error Reporting

Execute the cleaning pipeline and display detailed validation results for each column.

In [42]:
# pd.set_option('future.no_silent_downcasting', True) # error por el fill na
df_clean, report = transform_df(df, validation_rules)
print("Cleaning Succesfull!!")

for column, errors in report.items():
    print(column, errors)

Cleaning Succesfull!!
Reading Value {'out_of_range_n': np.int64(35), 'null_values_n': np.int64(7)}
Previous Reading Value {'out_of_range_n': np.int64(5), 'null_values_n': np.int64(7)}
Final Billing {'out_of_range_n': np.int64(0), 'null_values_n': np.int64(17)}
Type of Contract {'null_values_c': np.int64(0), 'invalid_values_c': np.int64(86)}
Reading Validity {'null_values_c': np.int64(0), 'invalid_values_c': np.int64(500)}
Certification on the ERP {'null_values_c': np.int64(0), 'invalid_values_c': np.int64(500)}
Reading Frequency {'null_values_c': np.int64(0), 'invalid_values_c': np.int64(0)}


## 6. Data Export

Save the cleaned and validated dataset for future analysis and reporting.

In [43]:
df_clean.to_csv("datasets/cleanData.csv", index=False)